# scVI integration and cell annotation


## Imports

In [1]:
from __future__ import annotations
from pathlib import Path
import gc
from cap.figures import (
    CONDITION_ORDER,
    CONDITION_PALETTE,
    configure_figure_style,
    save_figure,
    save_umap_figure,
)
from cap.paths import get_paths
from cap.provenance import (
    sha256_file,
    validate_frozen_inputs,
    write_run_provenance,
)


In [2]:
import hashlib
import json
import os


In [3]:
from matplotlib.colors import to_hex
from matplotlib.colors import to_rgba
from matplotlib.lines import Line2D


In [4]:
from sklearn.decomposition import PCA
import anndata as ad
import matplotlib as mpl


In [5]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [6]:
import scanpy as sc
import scipy.sparse as sp
import scvi


In [7]:
import seaborn as sns
import session_info


## Paths and reference genes

In [8]:
# Use a new CAP_RUN_ID for each complete pipeline rerun.
paths = get_paths()
paths.validate_base_inputs()
validate_frozen_inputs(paths)


In [9]:
notebook_path = paths.root / "notebooks" / "002_Celltype_ID.ipynb"
paths.require_file(notebook_path, "002 notebook")

WindowsPath('D:/IdeaProjects/CAP_Submission/CAP_Analysis/notebooks/002_Celltype_ID.ipynb')

In [10]:
stage_dir = paths.ensure_stage("02_cell_annotation")

In [11]:
preprocessing_dir = paths.stage_dir("01_preprocessing")
paths.require_dir(preprocessing_dir, "01_preprocessing stage")


FileNotFoundError: Missing 01_preprocessing stage: D:\CAP_Data\runs\preprocessing_test_007\01_preprocessing

In [ ]:
cohort_file = paths.root / "config" / "cohort_release_2026-11-13.csv"


In [ ]:
decision_file = paths.frozen_dir / "cell_annotation_decisions.json"
policy_file = paths.frozen_dir / "gene_policy.json"
celltypist_model_file = paths.celltypist_model_file


In [ ]:
# Fixed gene annotations and gene sets, not cell labels or analysis results.
policy_sha256 = "c4213d53e2d3209312e7781c73ed20f660e0f256042956742693c0e0375926a6"
os.environ["RPY2_CFFI_MODE"] = "ABI"


In [ ]:
paths.require_file(cohort_file, "frozen cohort release")
paths.require_file(decision_file, "frozen cell-annotation decisions")
paths.require_file(policy_file, "frozen gene policy")

with decision_file.open(encoding="utf-8") as handle:
    frozen_decisions = json.load(handle)

required_decision_keys = {
    "decision_timing",
    "run_reference_annotation",
    "excluded_patients",
    "cluster_labels",
    "low_quality_clusters",
}
missing_decision_keys = required_decision_keys.difference(frozen_decisions)
if missing_decision_keys:
    raise KeyError(
        f"Frozen decision file is missing: {sorted(missing_decision_keys)}"
    )

run_reference_annotation = frozen_decisions["run_reference_annotation"]
if type(run_reference_annotation) is not bool:
    raise TypeError("run_reference_annotation must be a frozen boolean decision")

if run_reference_annotation:
    paths.require_file(celltypist_model_file, "CellTypist reference model")

cohort_sha256 = sha256_file(cohort_file)
celltypist_model_sha256 = (
    sha256_file(celltypist_model_file) if run_reference_annotation else None
)


In [ ]:
if stage_dir.resolve() == preprocessing_dir.resolve():
    raise ValueError("02_cell_annotation must be separate from 01_preprocessing.")
if any(stage_dir.iterdir()):
    raise FileExistsError(
        f"{stage_dir} is not empty. Use a fresh CAP_RUN_ID; "
        "existing stage outputs are immutable."
    )


In [ ]:
head_dir = preprocessing_dir / "h5ad" / "head"
qc_input_dir = preprocessing_dir / "tables" / "by_sample"


In [ ]:
save_dir = stage_dir / "h5ad"
model_dir = stage_dir / "models"
fig_dir = stage_dir / "figures"
df_dir = stage_dir / "tables"


In [ ]:
for directory in (save_dir, model_dir, fig_dir, df_dir):
    directory.mkdir(parents=True, exist_ok=True)

input_checksums = {
    "cohort_release_2026-11-13.csv": cohort_sha256,
    "cell_annotation_decisions.json": sha256_file(decision_file),
    "gene_policy.json": sha256_file(policy_file),
    "frozen_inputs.json": sha256_file(paths.manifest_dir / "frozen_inputs.json"),
    "celltypist_model_file": celltypist_model_sha256,
}

(stage_dir / "input_checksums.json").write_text(
    json.dumps(input_checksums, indent=2) + "\n", encoding="utf-8")

write_run_provenance( paths, stage_dir, notebook_path=notebook_path, input_checksums=input_checksums)



In [ ]:
def load_gene_policy(path=policy_file):
    path = Path(path)
    payload = path.read_bytes()
    digest = hashlib.sha256(payload).hexdigest()
    if digest != policy_sha256:
        raise ValueError(
            "Gene-policy checksum differs from the reviewed source snapshot"
        )
    policy = json.loads(payload)
    policy["sha256"] = digest
    return policy


In [ ]:
gene_policy = load_gene_policy()


## Figure settings

The shared `cap.figures` configuration defines the publication style and SVG/PNG export behavior. UMAP exports rasterize dense point collections inside SVG files while retaining vector axes, labels, legends, and annotations.


In [ ]:
configure_figure_style()


In [ ]:
sc.settings.figdir = fig_dir
sc.settings.verbosity = 1
np.random.seed(0)


In [ ]:
# Preserve aliases used before condition labels are standardised.
condition_palette = {
    "HER2": CONDITION_PALETTE["HER2+"],
    "HER2+": CONDITION_PALETTE["HER2+"],
    "TN": CONDITION_PALETTE["TN"],
    "ER": CONDITION_PALETTE["ER+"],
    "ER+": CONDITION_PALETTE["ER+"],
    "N2": CONDITION_PALETTE["N"],
    "N": CONDITION_PALETTE["N"],
}


In [ ]:
condition_order = list(CONDITION_ORDER)
condition_rank = {condition: index for index, condition in enumerate(condition_order)}


In [ ]:
# SVG font handling is defined by the shared figure configuration.


In [ ]:
import logging
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)


## Cohort and preprocessed libraries

The cohort table and post-QC count matrices come from the selected 000/001
folder. The study's explicit MH0114 exclusion is retained. No integrated atlas,
trained scVI model, cell-label table or historical exclusion table is loaded.

In [ ]:
samples = pd.read_csv(cohort_file)


In [ ]:
excluded_patients = frozen_decisions["excluded_patients"]


In [ ]:
samples["include_before_patient_exclusion"] = samples["include"]


In [ ]:
patient_exclusion = samples["patient"].isin(excluded_patients)
samples.loc[patient_exclusion, "include"] = False
samples.loc[patient_exclusion, "exclusion_reason"] = samples.loc[
    patient_exclusion, "patient"
].map(excluded_patients)


In [ ]:
wanted = samples.loc[samples["include"], "sample_id"].tolist()
assert len(samples) == 69 and len(wanted) == (43 if excluded_patients else 45)
assert samples.loc[samples["include"], "patient"].nunique() == (41 if excluded_patients else 42)


In [ ]:
missing_libraries = [
    sample for sample in wanted if not (head_dir / f"{sample}.h5ad").is_file()
]
if missing_libraries:
    raise FileNotFoundError(
        f"001 preprocessing outputs are missing for {len(missing_libraries)} libraries. "
        "Set head_dir to the retained 001 output folder, or generate them with 001 before running 002."
    )
qc_files = [qc_input_dir / sample_id / "qc_summary.csv" for sample_id in wanted]
missing_qc = [str(f) for f in qc_files if not f.is_file()]
assert qc_files, "No samples selected for QC summary"
assert not missing_qc, f"Missing 001 QC tables in the configured run: {missing_qc}"
print(f"{len(wanted)} samples expected")


In [ ]:
adatas = []


In [ ]:
for sample_id in wanted:
    path = head_dir / f"{sample_id}.h5ad"
    assert path.exists(), f"001 output missing: {sample_id}"
    a = sc.read_h5ad(path)
    # Only post-QC libraries are valid inputs at this stage.
    derived_fields = {
        "scVI_leiden", "cl_annotation", "epithelial_candidate",
        "epithelial_primary", "epithelial_uncertain", "reference_cluster",
        "annotation_review_status", "annotation_review_subcluster",
        "celltypist_pal_label", "celltypist_pal_conf",
    }
    inherited_fields = derived_fields.intersection(a.obs.columns)
    if inherited_fields or "X_scVI" in a.obsm or "annotation_mapping" in a.uns:
        raise ValueError(f"Expected a 001 post-QC input, not an annotated/integrated object: {path}")
    a.obs["sample"] = sample_id
    adatas.append(a)


In [ ]:
assert len(adatas) == len(wanted), f"loaded {len(adatas)}, expected {len(wanted)}"
print(f"{len(adatas)} loaded, {sum((a.n_obs for a in adatas)):,} cells")


In [ ]:
deviance = pd.DataFrame({s: a.var["deviance"] for s, a in zip(wanted, adatas)})


## Concatenation and QC metrics

In [ ]:
adata = ad.concat(
    adatas, join="inner", label="batch", keys=wanted, index_unique="-", merge="same"
)


In [ ]:
# Each library has its own PCA basis; pooled embeddings are computed below.
adata.obsm.clear()
adata.obsp.clear()

In [ ]:
adata.obs["condition"] = adata.obs["sample"].str.split("_").str[1].str.split("-").str[0]
adata.obs["patient"] = adata.obs["sample"].map(
    dict(zip(samples.sample_id, samples.patient))
)


In [ ]:
assert not adata.obs["patient"].astype(str).isin(excluded_patients).any()
adata.uns["cohort_selection"] = {
    "excluded_patients": list(excluded_patients),
    "exclusion_reasons": dict(excluded_patients),
    "decision_timing": frozen_decisions["decision_timing"],
    "cohort": "donor-exclusion primary" if excluded_patients else "full-cohort sensitivity",
    "n_source_libraries": int(len(samples)),
    "n_selected_libraries": int(len(wanted)),
    "n_selected_patients": int(samples.loc[samples["include"], "patient"].nunique()),
}


In [ ]:
samples.to_csv(df_dir / "cohort_samples.csv", index=False)


In [ ]:
(df_dir / "cohort_selection.json").write_text(
    json.dumps(adata.uns["cohort_selection"], indent=2) + "\n"
)


In [ ]:
def feature_identities(var, policy):
    """Assign frozen HGNC IDs by stable matrix IDs, without changing var_names."""
    if not var.index.is_unique or "gene_id" not in var:
        raise ValueError(
            "Unique feature names and original Ensembl gene_id values are required"
        )
    ids = var["gene_id"].astype(str).str.split(".").str[0]
    canonical = {}
    for records in policy["symbols"].values():
        for record in records:
            if record["ensembl_id"]:
                canonical.setdefault(record["ensembl_id"], set()).add(record["hgnc_id"])
    values = []
    for identifier in ids:
        identity = policy["feature_hgnc_ids"].get(identifier)
        if identity is None:
            candidates = canonical.get(identifier, set())
            identity = next(iter(candidates)) if len(candidates) == 1 else ""
        values.append(identity)
    return pd.Series(values, index=var.index, name="hgnc_id")


In [ ]:
def apply_gene_flags(adata, policy):
    """Replace broad QC prefixes with reviewed family memberships."""
    identity = feature_identities(adata.var, policy)
    audit = pd.DataFrame(
        {
            "feature_gene": adata.var_names.astype(str),
            "feature_id": adata.var["gene_id"].astype(str).to_numpy(),
            "hgnc_id": identity.to_numpy(),
        }
    )
    for flag in ["mt", "ribo", "hb"]:
        old = (
            adata.var[flag].astype(bool).to_numpy()
            if flag in adata.var
            else np.zeros(adata.n_vars, bool)
        )
        new = identity.isin(policy["families"][flag]).to_numpy()
        adata.var[flag] = new
        audit[flag + "_before"] = old
        audit[flag + "_after"] = new
    return audit


In [ ]:
def refresh_ribosomal_qc(adata, layer="counts"):
    """Refresh the ribosomal diagnostic without changing cell-QC decisions."""
    matrix = adata.layers[layer]
    total = np.asarray(matrix.sum(axis=1)).ravel()
    ribosomal = np.asarray(
        matrix[:, adata.var["ribo"].to_numpy(bool)].sum(axis=1)
    ).ravel()
    adata.obs["total_counts_ribo"] = ribosomal
    adata.obs["log1p_total_counts_ribo"] = np.log1p(ribosomal)
    adata.obs["pct_counts_ribo"] = np.divide(
        100 * ribosomal, total, out=np.zeros_like(total, dtype=float), where=total > 0
    )


In [ ]:
gene_flag_audit = apply_gene_flags(adata, gene_policy)


In [ ]:
refresh_ribosomal_qc(adata)


In [ ]:
gene_flag_audit.to_csv(df_dir / "atlas_gene_flags.csv", index=False)


In [ ]:
d = adata.layers["counts"].data
assert np.allclose(d, np.round(d)), "counts layer is not integer"


In [ ]:
del adatas
gc.collect()


## Gene filtering and variable features

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
print("genes after filter:", adata.n_vars)


In [ ]:
for g in ["ROR1", "ROR2", "EGFR", "ERBB2", "EPCAM"]:
    assert g in adata.var_names, f"{g} was filtered out"


In [ ]:
adata.var["binomial_deviance"] = deviance.reindex(adata.var_names).max(axis=1)
assert adata.var["binomial_deviance"].notna().all()


In [ ]:
def resolve_signature(genes, var, policy):
    """Return unique available feature names and a complete source-member audit.

    Exact original features remain preferred when consistent with the frozen
    identity. Otherwise a unique stable-ID/HGNC match is required. Ambiguities
    stop the run; absent source members remain explicit audit rows.
    """
    identities = feature_identities(var, policy)
    ids = var["gene_id"].astype(str).str.split(".").str[0]
    rows, selected = ([], [])
    for gene in dict.fromkeys(map(str, genes)):
        candidates = policy["symbols"].get(gene, [])
        hgnc_ids = {r["hgnc_id"] for r in candidates}
        expected_ids = {r["ensembl_id"] for r in candidates if r["ensembl_id"]}
        chosen, method = (None, "unavailable")
        if gene in var.index:
            identity = identities.loc[gene]
            if hgnc_ids and identity and (identity not in hgnc_ids):
                raise ValueError(
                    f"Exact feature conflicts with source identity: {gene}"
                )
            chosen, method = (gene, "exact_original_feature")
        else:
            direct = var.index[ids.isin(expected_ids)]
            matches = direct if len(direct) else var.index[identities.isin(hgnc_ids)]
            if len(matches) > 1:
                raise ValueError(
                    f"Ambiguous source-to-feature mapping: {gene}: {list(matches)}"
                )
            if len(matches) == 1:
                chosen = str(matches[0])
                method = (
                    "verified_Ensembl_ID" if len(direct) else "frozen_HGNC_identity"
                )
        duplicate = chosen is not None and chosen in selected
        if chosen is not None and (not duplicate):
            selected.append(chosen)
        rows.append(
            {
                "source_gene": gene,
                "feature_gene": chosen or "",
                "feature_id": str(ids.loc[chosen]) if chosen is not None else "",
                "hgnc_id": identities.loc[chosen] if chosen is not None else "",
                "method": (
                    "duplicate_feature_not_double_weighted" if duplicate else method
                ),
                "used": chosen is not None and (not duplicate),
            }
        )
    return (selected, pd.DataFrame(rows))


In [ ]:
def cell_cycle_genes(var, policy):
    s_genes, s_audit = resolve_signature(policy["cell_cycle"]["s.genes"], var, policy)
    g2m_genes, g_audit = resolve_signature(
        policy["cell_cycle"]["g2m.genes"], var, policy
    )
    return (
        s_genes,
        g2m_genes,
        pd.concat(
            [s_audit.assign(phase="S"), g_audit.assign(phase="G2M")], ignore_index=True
        ),
    )


In [ ]:
s_genes, g2m_genes, cycle_gene_audit = cell_cycle_genes(adata.var, gene_policy)


In [ ]:
def stamp_gene_policy(adata, policy, stage, cell_cycle_scored=None):
    previous = dict(adata.uns.get("gene_policy", {}))
    previous.update(
        {"version": policy["version"], "sha256": policy["sha256"], "stage": stage}
    )
    if cell_cycle_scored is not None:
        previous["cell_cycle_scored"] = bool(cell_cycle_scored)
    adata.uns["gene_policy"] = previous


In [ ]:
def score_cell_cycle_by_sample(adata, policy, sample_key="sample", seed=0):
    """Refresh phase on post-QC raw counts, normalised per cell, one library at a time."""
    import anndata as ad
    import scanpy as sc

    s_genes, g2m_genes, audit = cell_cycle_genes(adata.var, policy)
    if not audit.used.all():
        raise ValueError(
            "The reviewed 43/54 cell-cycle panel is incomplete in this input"
        )
    matrix = adata.layers["counts"]
    for sample in adata.obs[sample_key].astype(str).unique():
        mask = adata.obs[sample_key].astype(str).eq(sample).to_numpy()
        tmp = ad.AnnData(
            X=matrix[mask].copy(),
            obs=pd.DataFrame(index=adata.obs_names[mask]),
            var=adata.var.copy(),
        )
        sc.pp.normalize_total(tmp, target_sum=10000.0)
        sc.pp.log1p(tmp)
        sc.tl.score_genes_cell_cycle(
            tmp, s_genes=s_genes, g2m_genes=g2m_genes, random_state=seed, use_raw=False
        )
        for column in ["S_score", "G2M_score"]:
            adata.obs.loc[tmp.obs_names, column] = tmp.obs[column].to_numpy(float)
        if "phase" not in adata.obs:
            adata.obs["phase"] = ""
        else:
            adata.obs["phase"] = adata.obs["phase"].astype(object)
        adata.obs.loc[tmp.obs_names, "phase"] = tmp.obs.phase.astype(str).to_numpy()
        del tmp
    adata.obs["phase"] = pd.Categorical(
        adata.obs["phase"], categories=["G1", "S", "G2M"]
    )
    stamp_gene_policy(adata, policy, "atlas", cell_cycle_scored=True)
    return audit


In [ ]:
cycle_gene_audit = score_cell_cycle_by_sample(adata, gene_policy, seed=0)


In [ ]:
cycle_gene_audit.to_csv(df_dir / "atlas_cell_cycle_genes.csv", index=False)


In [ ]:
exclude = adata.var[["mt", "ribo", "hb"]].any(axis=1).to_numpy()
exclude |= adata.var_names.isin(s_genes + g2m_genes)
exclude |= (
    feature_identities(adata.var, gene_policy)
    .isin(gene_policy["families"]["immunoglobulin"])
    .to_numpy()
)


In [ ]:
ranked = adata.var.loc[~exclude, "binomial_deviance"].sort_values(ascending=False)


In [ ]:
adata.var["highly_variable"] = adata.var_names.isin(ranked.index[:2000])
assert adata.var["highly_variable"].sum() == 2000


## Structure before integration

In [ ]:
adata_pre = adata[:, adata.var["highly_variable"]].copy()
adata_pre.X = adata_pre.layers["log1p"]
print(f"Shape after HVG subset: {adata_pre.shape}")


In [ ]:
sc.pp.scale(adata_pre, max_value=10, zero_center=False)


In [ ]:
sc.tl.pca(adata_pre, n_comps=20, zero_center=False, random_state=0)


In [ ]:
sc.pp.neighbors(adata_pre, use_rep="X_pca", n_pcs=20, n_neighbors=15, random_state=0)


In [ ]:
sc.tl.umap(adata_pre, random_state=0)


In [ ]:
assert len(qc_files) == len(wanted), "QC preflight does not match selected libraries"


In [ ]:
qc = pd.concat([pd.read_csv(path) for path in qc_files], ignore_index=True)

if "sample_id" in qc.columns and "sample" not in qc.columns:
    qc = qc.rename(columns={"sample_id": "sample"})

if ("miqc_mt_cutoff" in qc.columns and "max_retained_pct_mt" not in qc.columns):
    qc = qc.rename(columns={"miqc_mt_cutoff": "max_retained_pct_mt"})

assert qc["sample"].is_unique
assert set(qc["sample"]) == set(wanted), (
    "QC tables do not match the selected libraries"
)

In [ ]:
qc = qc.sort_values(["condition", "sample"]).reset_index(drop=True)


In [ ]:
qc.to_csv(df_dir / "qc_summary.csv", index=False)


In [ ]:
print(
    qc.groupby("condition")[["n_cells_final", "max_retained_pct_mt", "pct_doublets"]]
    .agg(["count", "sum", "median"])
    .round(1)
    .to_string()
)


In [ ]:
display(qc)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
sns.barplot(
    data=qc,
    x="sample",
    y="n_cells_final",
    hue="condition",
    palette=condition_palette,
    dodge=False,
    ax=ax,
)
ax.set(xlabel="", ylabel="cells after QC")
ax.tick_params(axis="x", rotation=90, labelsize=6)
save_figure(fig, fig_dir / "cells_per_sample", close=False, overwrite=False)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
sns.barplot(
    data=qc,
    x="sample",
    y="max_retained_pct_mt",
    hue="condition",
    palette=condition_palette,
    dodge=False,
    ax=ax,
)
ax.set(xlabel="", ylabel="Maximum retained\nmitochondrial fraction (%)")
ax.tick_params(axis="x", rotation=90, labelsize=6)
save_figure(fig, fig_dir / "maximum_retained_mitochondrial_fraction_per_sample", close=False, overwrite=False)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
sns.barplot(
    data=qc,
    x="sample",
    y="pct_doublets",
    hue="condition",
    palette=condition_palette,
    dodge=False,
    ax=ax,
)
ax.set(xlabel="", ylabel="doublets removed (%)")
ax.tick_params(axis="x", rotation=90, labelsize=6)
save_figure(fig, fig_dir / "doublets_per_sample", close=False, overwrite=False)
plt.show()


In [ ]:
sc.pl.umap(adata_pre, color="sample", show=False)
plt.tight_layout()
save_umap_figure(plt.gcf(), fig_dir / "umap_before_integration", close=False, overwrite=False)
plt.show()


In [ ]:
del adata_pre
gc.collect()


## scVI integration

In [ ]:
assert not (
    feature_identities(adata.var, gene_policy).isin(
        gene_policy["families"]["immunoglobulin"]
    )
    & adata.var["highly_variable"]
).any()


In [ ]:
adata_scvi = adata[:, adata.var["highly_variable"]].copy()


In [ ]:
scvi.settings.seed = 0
scvi.settings.num_workers = 4


In [ ]:
scvi.model.SCVI.setup_anndata(adata_scvi, layer="counts", batch_key="sample")


To Train a new model

In [ ]:
model = scvi.model.SCVI(adata_scvi, n_latent=30, n_layers=2, gene_likelihood="nb")


In [ ]:
model.train(
    max_epochs=400,
    accelerator="cpu",
    batch_size=1024,
    early_stopping=True,
    early_stopping_patience=20,
    early_stopping_min_delta=0.5,
)
print(f"stopped after {model.history['elbo_train'].shape[0]} epochs")

In [ ]:
model.save(str(model_dir), overwrite=False)

In [ ]:
model.history["elbo_train"].to_csv(df_dir / "scvi_atlas_history.csv")

In [ ]:
adata_scvi.obsm["X_scVI"] = model.get_latent_representation()
assert not np.isnan(adata_scvi.obsm["X_scVI"]).any()

In [ ]:
del model
gc.collect()

In [ ]:
adata.obsm["X_scVI"] = adata_scvi.obsm["X_scVI"]

In [ ]:
pd.Series(adata_scvi.var_names.astype(str)).to_csv(
    model_dir / "var_names.csv",
    index=False,
    header=False,
)

In [ ]:
del adata_scvi
gc.collect()

## Neighbours, clusters and UMAP

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=30, random_state=0)


In [ ]:
sc.tl.leiden(adata,
             key_added="scVI_leiden", resolution=1,
    flavor="igraph", n_iterations=2, directed=False, random_state=0)


In [ ]:
sc.tl.umap(adata, random_state=0)


In [ ]:
r = np.corrcoef(adata.obsm["X_scVI"][:, 0], adata.obs["total_counts"])[0, 1]
print(f"scVI latent dim 1 vs total counts: r = {r:+.2f}")


In [ ]:
pca = PCA(n_components=2, random_state=0).fit_transform(adata.obsm["X_scVI"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
conditions = adata.obs["condition"].astype("category")
for cond, color in condition_palette.items():
    mask = conditions == cond
    axes[0].scatter(pca[mask, 0], pca[mask, 1], s=1, label=cond, color=color, alpha=0.5)
axes[0].set_title("Condition")
axes[0].legend(markerscale=5)
patients = adata.obs["sample"].astype("category")
colors = plt.cm.tab20(np.linspace(0, 1, len(patients.cat.categories)))
for i, pat in enumerate(patients.cat.categories):
    mask = patients == pat
    axes[1].scatter(
        pca[mask, 0], pca[mask, 1], s=1, label=pat, color=colors[i], alpha=0.5
    )
axes[1].set_title("Sample ID")
axes[1].legend(markerscale=5, fontsize=6, ncol=2)
del pca
gc.collect()
plt.tight_layout()
save_figure(fig, fig_dir / "scvi_latent_pca", close=False, overwrite=False)
plt.show()


In [ ]:
conn = adata.obsp["connectivities"].tocsr()
codes = adata.obs["sample"].astype("category").cat.codes.to_numpy()
same = codes[conn.indices] == np.repeat(codes, np.diff(conn.indptr))


In [ ]:
n_nb = np.maximum(np.diff(conn.indptr), 1)
frac_same = np.add.reduceat(same, conn.indptr[:-1]) / n_nb


In [ ]:
p = adata.obs["sample"].value_counts(normalize=True).to_numpy()
expected = (p**2).sum()


In [ ]:
print(f"neighbours from the same sample : {frac_same.mean():.3f}")
print(f"expected if perfectly mixed     : {expected:.3f}")
print(f"ratio                           : {frac_same.mean() / expected:.1f}x")


In [ ]:
adata.write_h5ad(save_dir / "project_integrated.h5ad", compression="gzip")


In [ ]:
sc.pl.umap(
    adata,
    color=["sample", "condition", "patient", "scVI_leiden"],
    wspace=0.4,
    show=False,
)
save_umap_figure(plt.gcf(), fig_dir / "umap_sample_condition_patient_leiden", close=False, overwrite=False)
plt.show()


## Specimen labels

In [ ]:
adata.obs["condition"] = adata.obs["condition"].str.replace("^ER$", "ER+", regex=True)
adata.obs["condition"] = adata.obs["condition"].str.replace(
    "^HER2$", "HER2+", regex=True
)
adata.obs["condition"] = pd.Categorical(
    adata.obs["condition"], categories=condition_order, ordered=True
)

In [ ]:
unexpected = set(adata.obs["condition"].unique()) - set(condition_order)
assert not unexpected, f"Unexpected conditions: {unexpected}"

In [ ]:
adata.obs["sample_short"] = (
    adata.obs["sample"].astype(str)
    .str.strip().str.replace(".h5ad", "", regex=False).str.split("_", n=1).str[-1]
    .str.replace("^(TN|ER|HER2|N)-", "", regex=True).astype("category"))

## Cluster markers

In [ ]:
adata.layers["lognorm"] = adata.layers["log1p"]


In [ ]:
sc.tl.rank_genes_groups(
    adata, groupby="scVI_leiden", layer="lognorm", method="wilcoxon")


In [ ]:
degs = pd.DataFrame(adata.uns["rank_genes_groups"]["names"]).head(60)


In [ ]:
degs.to_csv(df_dir / "cluster_degs.csv", index=False)
display(degs)


In [ ]:
sc.pl.rank_genes_groups_dotplot(
    adata, n_genes=5, swap_axes=False, dendrogram=False, show=False)
save_figure(plt.gcf(), fig_dir / "dotplot_cluster_degs", close=False, overwrite=False)
plt.show()


In [ ]:
sc.pl.umap(adata, color="scVI_leiden", legend_loc="on data", show=False)
save_umap_figure(plt.gcf(), fig_dir / "umap_cluster_numbers", close=False, overwrite=False)
plt.show()


## Lineage markers and cluster QC

In [ ]:
broad_markers = {
    "T Cells": ["PTPRC", "CD3D", "CD3E", "TRAC"],
    "NK/T Cells": ["NKG7", "GNLY", "KLRD1", "KLRF1"],
    "B Cells": ["CD79A", "CD74", "MS4A1"],
    "Macrophages": ["CD68", "CD14", "ITGAM"],
    "Fibroblasts": ["COL1A1", "CXCL12", "MYL9"],
    "Plasma Cells": ["MZB1", "JCHAIN", "IGHG1"],
    "Endothelial Cells": ["VWF", "CDH5", "PECAM1"],
    "Mast Cells": ["KIT", "CPA3", "TPSAB1"],
    "Dendritic Cells": ["ITGAX", "HLA-DRA", "CD83"],
    "Epithelial Cells": ["EPCAM", "KRT19", "KRT18"],
    "Basal": ["KRT14", "KRT17", "OXTR"],
}


In [ ]:
broad_markers_in_data = {
    cell_type: [gene for gene in genes if gene in adata.var_names]
    for cell_type, genes in broad_markers.items()
}


In [ ]:
detailed_markers = {
    "T cells": ["PTPRC", "CD3D", "CD3E", "CD3G", "TRAC", "IL7R", "CD8A"],
    "NK cells": ["KLRF1", "NCR1", "FCER1G", "TYROBP", "NKG7", "GNLY", "KLRD1"],
    "B cells": ["CD79A", "CD79B", "MS4A1", "CD74"],
    "Plasma cells": ["MZB1", "JCHAIN", "IGHG1", "XBP1", "DERL3"],
    "Macrophages": ["CD68", "CD14", "ITGAM", "LYZ", "AIF1", "CSF1R"],
    "SPP1+ macrophages": ["SPP1", "ACP5", "MMP9", "CTSK"],
    "Dendritic cells": ["ITGAX", "CD1C", "CLEC9A", "CD83"],
    "pDC": ["IRF7", "IRF8", "TCF4", "GZMB", "LILRA4", "PLAC8"],
    "Mast cells": ["KIT", "CPA3", "TPSAB1", "TPSB2", "MS4A2"],
    "Fibroblasts": ["COL1A1", "COL1A2", "COL3A1", "DCN", "LUM", "PDGFRA"],
    "Perivascular": ["ACTA2", "RGS5", "PDGFRB", "MYH11", "TAGLN"],
    "Endothelial": ["PECAM1", "VWF", "CDH5", "CLDN5"],
    "Lymphatic": ["CCL21", "PDPN", "PROX1", "LYVE1"],
    "Epithelial": ["EPCAM", "KRT8", "KRT18", "KRT19"],
    "Basal": ["KRT14", "KRT5", "KRT17", "TP63", "MYLK", "OXTR"],
    "Luminal progenitor": ["ALDH1A3", "KIT", "ELF5", "SLPI", "LTF", "KRT15"],
    "Mature luminal": ["ESR1", "PGR", "FOXA1", "ANKRD30A", "AREG", "TFF1", "GATA3"],
    "Ciliated": ["FOXJ1", "TPPP3", "CAPS", "FAM183A", "RSPH1", "PIFO"],
    "Cycling": ["MKI67", "TOP2A", "CCNB1"],
}


In [ ]:
available = {k: [g for g in v if g in adata.var_names] for k, v in detailed_markers.items()}
missing = {
    k: [g for g in v if g not in adata.var_names] for k, v in detailed_markers.items()
}
print("missing:", {k: v for k, v in missing.items() if v})


In [ ]:
sc.pl.dotplot(
    adata,
    available,
    groupby="scVI_leiden",
    layer="log1p",
    standard_scale="var",
    smallest_dot=10,
    dendrogram=False,
    figsize=[26, 10],
    show=False,
)
save_figure(plt.gcf(), fig_dir / "dotplot_markers_all", close=False, overwrite=False)
plt.show()


In [ ]:
print(
    adata.obs.groupby("scVI_leiden")[
        ["pct_counts_mt", "pct_counts_ribo", "n_genes_by_counts"]
    ]
    .median()
    .round(1)
    .to_string()
)


In [ ]:
b_plasma_cts = ['T Cells', 'B Cells', 'Macrophages', 'Fibroblasts', 'Plasma Cells', 'Endothelial Cells', 'Mast Cells', 'Dendritic Cells', 'NK/T Cells']


In [ ]:
b_plasma_markers = {
    ct: [m for m in ct_markers if m in adata.var.index]
    for ct, ct_markers in broad_markers_in_data.items()
    if ct in b_plasma_cts
}


In [ ]:
sc.pl.dotplot(
    adata,
    groupby="scVI_leiden",
    var_names=b_plasma_markers,
    layer="log1p",
    standard_scale="var",
    show=False,
)
save_figure(plt.gcf(), fig_dir / "dotplot_broad_lineages", close=False, overwrite=False)
plt.show()


In [ ]:
marker_genes = {
    "T Cells": ["PTPRC", "CD3D", "CD3E", "TRAC"],
    "NK/T Cells": ["NKG7", "GNLY", "KLRD1", "KLRF1"],
    "B Cells": ["CD79A", "MS4A1", "CD74"],
    "Plasma Cells": ["MZB1", "JCHAIN", "IGHG1"],
    "Macrophages": ["CD68", "CD14", "ITGAM"],
    "Dendritic Cells": ["ITGAX", "HLA-DRA", "CD83"],
    "Mast Cells": ["KIT", "CPA3", "TPSAB1"],
    "Fibroblasts": ["COL1A1", "CXCL12", "MYL9"],
    "Endothelial Cells": ["VWF", "CDH5", "PECAM1"],
    "Epithelial Cells": ["EPCAM", "KRT19", "KRT18"],
    "Basal": ["KRT14", "KRT17", "OXTR"],
}


## Reference annotation

Supporting predictions use the original Ensembl IDs in `adata.var["gene_id"]`.
The supplied reference contains six epithelial/stromal classes and no immune
classes. Manual annotation uses cluster markers, differential expression and
QC alongside the reference predictions.


In [ ]:
# Reference-annotation setting is loaded from the frozen decision file.


In [ ]:
if run_reference_annotation:
    adata_norm = ad.AnnData(
        X=adata.layers["counts"].copy(), obs=adata.obs.copy(), var=adata.var.copy()
    )
    if not sp.issparse(adata_norm.X):
        adata_norm.X = sp.csr_matrix(adata_norm.X)
    elif not isinstance(adata_norm.X, sp.csr_matrix):
        adata_norm.X = adata_norm.X.tocsr()
    sc.pp.normalize_total(adata_norm, target_sum=10000.0)


In [ ]:
if run_reference_annotation:
    sc.pp.log1p(adata_norm)
    import gc
    gc.collect()


In [ ]:
if run_reference_annotation:
    # Preserve the original ID attached to each expression feature.
    gene_ids = adata_norm.var["gene_id"].astype("string")
    assert gene_ids.notna().all(), "Missing original gene IDs."
    assert gene_ids.is_unique, "Original gene IDs must be unique."
    assert gene_ids.str.fullmatch(r"ENSG[0-9]+").all(), "Expected unversioned Ensembl gene IDs."
    adata_norm.var["ensembl_id"] = gene_ids.astype(str)


In [ ]:
if run_reference_annotation:
    n_mapped = adata_norm.var["ensembl_id"].notna().sum()
    print(f"mapped {n_mapped:,} / {adata_norm.n_vars:,} genes to Ensembl IDs")


In [ ]:
if run_reference_annotation:
    import celltypist
    from celltypist import models
    mask = adata_norm.var["ensembl_id"].notna()


In [ ]:
if run_reference_annotation:
    adata_pal = adata_norm[:, mask].copy()
    adata_pal.var_names = adata_pal.var["ensembl_id"].astype(str)
    pred_pal = celltypist.annotate(
        adata_pal,
        model=models.Model.load(str(celltypist_model_file)),
        majority_voting=False,
    )


In [ ]:
if run_reference_annotation:
    del adata_pal
    gc.collect()
    del adata_norm


In [ ]:
if run_reference_annotation:
    # Assign predictions only after checking the cell order.
    assert pred_pal.predicted_labels.index.equals(adata.obs_names)
    adata.obs["celltypist_pal_label"] = pred_pal.predicted_labels[
        "predicted_labels"
    ].values
    adata.obs["celltypist_pal_conf"] = pred_pal.probability_matrix.max(axis=1).values


In [ ]:
if run_reference_annotation:
    fig, axes = plt.subplots(2, 2, figsize=(20, 18))
    axes = axes.flatten()
    sc.pl.umap(adata, color="condition", ax=axes[0], layer="log1p", show=False)
    axes[0].set_title("Condition")
    sc.pl.umap(adata, color="patient", ax=axes[1], layer="log1p", show=False)
    axes[1].set_title("Patient ID")
    sc.pl.umap(
        adata, color="celltypist_pal_label", ax=axes[2], layer="log1p", show=False
    )
    axes[2].set_title("Celltypist Label")
    sc.pl.umap(
        adata,
        color="celltypist_pal_conf",
        frameon=False,
        ax=axes[3],
        layer="log1p",
        show=False,
    )
    axes[3].set_title("Celltypist_Confidence")
    for ax in axes:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    save_umap_figure(fig, fig_dir / "umap_reference_annotation", close=False, overwrite=False)
    plt.show()


In [ ]:
if run_reference_annotation:
    cluster_reference = pd.crosstab(
        adata.obs["scVI_leiden"], adata.obs["celltypist_pal_label"], normalize="index"
    )
    cluster_reference.to_csv(df_dir / "cluster_annotation_reference_fractions.csv")
    display((100 * cluster_reference).round(1))


## Manual annotation


In [ ]:
cluster_qc = adata.obs.groupby("scVI_leiden", observed=True).agg(
    n_cells=("sample", "size"),
    median_genes=("n_genes_by_counts", "median"),
    median_pct_mt=("pct_counts_mt", "median"),
    median_pct_ribo=("pct_counts_ribo", "median"),
    median_doublet_score=("doublet_score", "median"),
)
display(cluster_qc.round(2))


In [ ]:
cl_annotation = frozen_decisions["cluster_labels"]


In [ ]:
low_quality_clusters = set(frozen_decisions["low_quality_clusters"])


In [ ]:
observed_clusters = set(adata.obs["scVI_leiden"].astype(str).unique())
assert set(cl_annotation) == observed_clusters, (
    "Review the current markers, fill cl_annotation for every cluster, "
    "then continue from the cl_annotation cell."
)


In [ ]:
model_dir = Path(
    r"CAP_Data\runs\preprocessing_test_003\02_cell_annotation\models"
)

saved_vars = pd.read_csv(
    model_dir / "var_names.csv",
    header=None,
).iloc[:, 0].astype(str)

# Remove a possible CSV header.
if saved_vars.iloc[0] in {"0", "var_names", "gene", "gene_name"}:
    saved_vars = saved_vars.iloc[1:]

saved_vars = pd.Index(saved_vars)
current_vars = pd.Index(adata_scvi.var_names.astype(str))

print("Saved features:", len(saved_vars))
print("Current features:", len(current_vars))
print("Same set:", set(saved_vars) == set(current_vars))
print("Same order:", saved_vars.equals(current_vars))
print("Missing:", list(saved_vars.difference(adata.var_names))[:20])

In [ ]:
adata.obs["cl_annotation"] = (
    adata.obs["scVI_leiden"].astype(str).map(cl_annotation).astype("category")
)


In [ ]:
assert adata.obs["cl_annotation"].notna().all()
assert low_quality_clusters.issubset(observed_clusters)


In [ ]:
adata.obs["epithelial_candidate"] = (
    adata.obs["cl_annotation"].astype(str).str.startswith("Epithelial")
)
adata.obs["epithelial_uncertain"] = adata.obs["epithelial_candidate"] & adata.obs[
    "scVI_leiden"
].astype(str).isin(low_quality_clusters)
adata.obs["epithelial_primary"] = (
    adata.obs["epithelial_candidate"] & ~adata.obs["epithelial_uncertain"]
)


In [ ]:
adata.uns["annotation_mapping"] = {
    "method": "manual",
    "cluster_key": "scVI_leiden",
    "cluster_labels": dict(cl_annotation),
    "low_quality_clusters": sorted(low_quality_clusters),
}


In [ ]:
cluster_qc["annotation"] = cluster_qc.index.astype(str).map(cl_annotation)
cluster_qc.to_csv(df_dir / "cluster_annotation_qc.csv")
display(cluster_qc.round(2))


## Cell metadata

In [ ]:
export_columns = [
    "sample",
    "patient",
    "condition",
    "scVI_leiden",
    "cl_annotation",
    "epithelial_candidate",
    "epithelial_primary",
    "epithelial_uncertain",
]

In [ ]:
adata.obs[export_columns].to_csv(df_dir / "cell_metadata.csv", index_label="barcode")


In [ ]:
adata.obs[export_columns].to_csv(df_dir / "barcode_table.csv", index_label="barcode")


In [ ]:
adata.obs.loc[adata.obs.epithelial_candidate, export_columns].to_csv(
    df_dir / "epithelial_inclusive_barcodes.csv.gz", index_label="barcode"
)


In [ ]:
adata.obs.loc[adata.obs.epithelial_primary, export_columns].to_csv(
    df_dir / "epithelial_barcodes.csv", index_label="barcode"
)


In [ ]:
adata.obs.loc[adata.obs.epithelial_uncertain, export_columns].to_csv(
    df_dir / "epithelial_uncertain_cells.csv.gz", index_label="barcode"
)


In [ ]:
adata.obs.groupby("condition", observed=True)[
    ["epithelial_candidate", "epithelial_primary", "epithelial_uncertain"]
].sum().to_csv(df_dir / "epithelial_identity_population_counts.csv")


## Epithelial inclusion sensitivity



Raw target counts are exported for the selected epithelium and the population
before manual quality exclusions. Statistical comparisons are performed in 003c.

In [ ]:
target_counts = pd.DataFrame(index=adata.obs_names)
for gene in ["ROR1", "EGFR", "EPCAM", "ERBB2"]:
    values = adata[:, gene].layers["counts"]
    target_counts[gene] = np.asarray(
        values.toarray() if sp.issparse(values) else values
    ).ravel()


In [ ]:
scope_masks = {
    "primary_epithelial": adata.obs.epithelial_primary,
    "including_uncertain_candidates": adata.obs.epithelial_candidate,
}


In [ ]:
scope_tables = []


In [ ]:
for min_umi in [1, 2]:
    measured = adata.obs[["sample", "patient", "condition"]].copy()
    measured["n_cells"] = 1
    measured["n_ror1"] = target_counts.ROR1.ge(min_umi)
    measured["n_egfr"] = target_counts.EGFR.ge(min_umi)
    measured["n_double_positive"] = measured.n_ror1 & measured.n_egfr
    measured["n_epcam_erbb2"] = target_counts.EPCAM.ge(
        min_umi
    ) & target_counts.ERBB2.ge(min_umi)
    for scope, mask in scope_masks.items():
        table = (
            measured.loc[mask]
            .groupby(["sample", "patient", "condition"], observed=True)[
                ["n_cells", "n_ror1", "n_egfr", "n_double_positive", "n_epcam_erbb2"]
            ]
            .sum()
            .reset_index()
        )
        table["scope"] = scope
        table["min_umi"] = min_umi
        scope_tables.append(table)


In [ ]:
identity_scope_counts = pd.concat(scope_tables, ignore_index=True)


In [ ]:
identity_scope_counts.to_csv(
    df_dir / "epithelial_identity_scope_counts_by_library.csv", index=False
)


In [ ]:
(df_dir / "manual_cell_annotation.json").write_text(
    json.dumps(adata.uns["annotation_mapping"], indent=2) + "\n"
)


In [ ]:
del target_counts, measured, scope_tables


In [ ]:
if run_reference_annotation:
    summary = (
        adata.obs.groupby("cl_annotation")["celltypist_pal_conf"]
        .agg(["count", "median"])
        .round(2)
        .sort_values("median")
    )
    print(summary.to_string())


In [ ]:
epi = adata.obs["cl_annotation"].astype(str).str.startswith("Epithelial")


In [ ]:
if run_reference_annotation:
    cross = pd.crosstab(
        adata.obs.loc[epi, "scVI_leiden"],
        adata.obs.loc[epi, "celltypist_pal_label"],
        normalize="index",
    )


In [ ]:
if run_reference_annotation:
    display((100 * cross).round(1))


In [ ]:
if run_reference_annotation:
    conf = adata.obs["celltypist_pal_conf"] > 0.5


In [ ]:
if run_reference_annotation:
    cross = pd.crosstab(
        adata.obs["scVI_leiden"],
        adata.obs["celltypist_pal_label"].astype(str).where(conf, "low confidence"),
    )


In [ ]:
if run_reference_annotation:
    display((100 * cross.div(cross.sum(axis=1), axis=0)).round(1))


In [ ]:
if run_reference_annotation:
    display(adata.obs.groupby("scVI_leiden")["celltypist_pal_conf"].apply(
        lambda x: (x < 0.5).mean() * 100
    ).round(1).sort_values(ascending=False).head(12))


In [ ]:
available = {k: [g for g in v if g in adata.var_names] for k, v in marker_genes.items()}


In [ ]:
marker_dotplot = sc.pl.dotplot(
    adata, available, groupby="cl_annotation", layer="log1p",
    standard_scale="var", smallest_dot=10, figsize=[22, 9],
    return_fig=True, show=False,
)
marker_dotplot.legend(
    width=2.5, size_title="Cells expressing\ngene (%)",
    colorbar_title="Scaled mean\nexpression",
)
marker_dotplot.make_figure()
save_figure(marker_dotplot.fig, fig_dir / "dotplot_celltype_annotation_markers", close=False, overwrite=False)
plt.show()


In [ ]:
epi = adata[adata.obs["cl_annotation"].astype(str).str.startswith("Epithelial")]
x = np.asarray(epi[:, "EPCAM"].layers["log1p"].todense()).ravel()
print(f"EPCAM detected in {100 * (x > 0).mean():.1f}% of the epithelial subset")


## Epithelial compartment

In [ ]:
plot_order = ["N", "TN", "HER2+", "ER+"]
condition_rank = {condition: i for i, condition in enumerate(plot_order)}
plot_idx = adata.obs["condition"].map(condition_rank).sort_values().index


In [ ]:
sc.pl.umap(
    adata[plot_idx],
    color="condition",
    palette=condition_palette,
    ncols=1,
    vmax="p99",
    show=False,
)
ax = plt.gca()
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)
ax.set_xlabel("UMAP1", fontsize=25)
ax.set_ylabel("UMAP2", fontsize=25)
ax.tick_params(axis="both", labelsize=12)
legend = ax.get_legend()
if legend is not None:
    plt.setp(legend.get_texts(), fontsize=15)
    for handle in legend.legend_handles:
        handle.set_sizes([70])
save_umap_figure(plt.gcf(), fig_dir / "umap_condition", close=False, overwrite=False)
plt.show()


In [ ]:
celltype_categories = list(
    pd.Categorical(adata.obs["cl_annotation"]).categories.astype(str)
)
celltypes = sorted(set(celltype_categories))
colors = plt.get_cmap("tab20b").colors


In [ ]:
assert len(celltypes) <= len(colors), "More cell types than tab20b colours"
celltype_palette = {label: to_hex(colors[i]) for i, label in enumerate(celltypes)}
adata.uns["cl_annotation_colors"] = [
    celltype_palette[label] for label in celltype_categories
]


In [ ]:
missing = set(adata.obs["cl_annotation"].astype(str).unique()) - set(celltype_palette)
assert not missing, f"palette missing: {missing}"
pd.Series(celltype_palette, name="colour").rename_axis("cell_type").to_csv(
    df_dir / "celltype_palette.csv"
)


In [ ]:
gc.collect()


In [ ]:
assert adata.obs["cl_annotation"].notna().all()


In [ ]:
epithelial_mask = adata.obs["epithelial_primary"].astype(bool)


In [ ]:
epithelial_obs = adata.obs.loc[
    epithelial_mask, ["sample", "condition", "cl_annotation", "scVI_leiden"]
].copy()


In [ ]:
print(f"Epithelial cells: {epithelial_mask.sum():,}")


In [ ]:
print("\nCell types:")
print(epithelial_obs["cl_annotation"].value_counts())


In [ ]:
print("\nConditions:")
print(epithelial_obs["condition"].value_counts())


In [ ]:
print("\nSamples:")
print(epithelial_obs["sample"].value_counts())


In [ ]:
if run_reference_annotation:
    pd.crosstab(adata.obs["scVI_leiden"], adata.obs["celltypist_pal_label"]).to_csv(
        df_dir / "cluster_vs_celltypist.csv"
    )


## Atlas composition

In [ ]:
sc.pl.umap(
    adata,
    color="cl_annotation",
    palette=celltype_palette,
    legend_loc="right margin",
    legend_fontsize=8,
    frameon=False,
    title="Cell type",
    show=False,
)
save_umap_figure(plt.gcf(), fig_dir / "atlas_celltype_umap", close=False, overwrite=False)
plt.show()


In [ ]:
comp = pd.crosstab(
    adata.obs["sample_short"], adata.obs["cl_annotation"], normalize="index"
)


In [ ]:
order = (
    adata.obs[["sample_short", "condition"]]
    .drop_duplicates()
    .assign(rank=lambda d: d["condition"].map({"N": 0, "TN": 1, "HER2+": 2, "ER+": 3}))
    .sort_values(["rank", "sample_short"])["sample_short"]
)
comp = comp.loc[order]


In [ ]:
cols = [c for c in celltype_palette if c in comp.columns] + [
    c for c in comp.columns if c not in celltype_palette
]
comp = comp[cols]


In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
comp.plot(
    kind="bar",
    stacked=True,
    width=0.85,
    ax=ax,
    color=[celltype_palette.get(c, "lightgrey") for c in comp.columns],
)
ax.set_xlabel("")
ax.set_ylabel("fraction of cells")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", labelsize=7, rotation=90)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False, fontsize=8)
ax.spines[["top", "right"]].set_visible(False)
save_figure(fig, fig_dir / "atlas_celltype_composition_by_sample", close=False, overwrite=False)
plt.show()


In [ ]:
comp.to_csv(df_dir / "celltype_fraction_per_sample.csv")


In [ ]:
sc.pl.umap(
    adata,
    color=["PTPRC", "COL1A1", "PECAM1", "EPCAM"],
    ncols=4,
    layer="log1p",
    cmap="RdYlBu_r",
    vmin=0,
    vmax="p99",
    size=8,
    frameon=False,
    show=False,
)
save_umap_figure(plt.gcf(), fig_dir / "atlas_lineage_marker_umaps", close=False, overwrite=False)
plt.show()


In [ ]:
genes_present = [
    g
    for g in sorted({g for v in marker_genes.values() for g in v})
    if g in adata.var_names
]


In [ ]:
adata_sub = adata[:, genes_present].copy()
x_matrix = adata_sub.layers["log1p"]


In [ ]:
df = pd.DataFrame(
    x_matrix.toarray() if hasattr(x_matrix, "toarray") else x_matrix,
    index=adata_sub.obs_names,
    columns=genes_present,
)
df["cluster"] = adata_sub.obs["scVI_leiden"].values
df["cell_type"] = adata_sub.obs["cl_annotation"].values


In [ ]:
df.index.name = "barcode"
tmp = df.reset_index()


In [ ]:
long_df = tmp.melt(
    id_vars=["barcode", "cluster", "cell_type"],
    var_name="gene",
    value_name="expression",
)


In [ ]:
long_df.to_csv(df_dir / "dotplot_long_format.csv", index=False)


In [ ]:
del adata_sub
gc.collect()


## Save annotated cells

In [ ]:
def _atomic_h5ad(adata, path):
    path = Path(path)
    temporary = path.with_suffix(".tmp.h5ad")
    adata.write_h5ad(temporary, compression="gzip")
    os.replace(temporary, path)


In [ ]:
if "epi" in globals():
    del epi
gc.collect()


In [ ]:
if getattr(adata, "isbacked", False):
        raise ValueError("Use the in-memory atlas for this export; no training is needed")


In [ ]:
assert adata.obs["cl_annotation"].notna().all()
assert adata.uns["annotation_mapping"]["method"] == "manual"


In [ ]:
save_dir.mkdir(parents=True, exist_ok=True)


In [ ]:
_atomic_h5ad(adata, save_dir / "project_annotated.h5ad")


In [ ]:
epi = adata[adata.obs.epithelial_primary.to_numpy(bool)].copy()
assert epi.obs["epithelial_primary"].all()


In [ ]:
_atomic_h5ad(epi, save_dir / "epithelial.h5ad")


In [ ]:
receipt = {"method": "manual", "n_atlas": adata.n_obs, "n_epithelial": epi.n_obs}
receipt.update(atlas_file="project_annotated.h5ad", primary_file="epithelial.h5ad", status="complete")


In [ ]:
(df_dir / "epithelial_identity_completion.json").write_text(
        json.dumps(receipt, indent=2) + "\n")


In [ ]:
print("primary epithelial:", epi.shape)


## Epithelial sample contribution

In [ ]:
contrib = (
    epi.obs.groupby(["condition", "sample_short"], observed=True)
    .size().rename("cells").reset_index())
contrib["rank"] = contrib["condition"].map(condition_rank)
contrib = contrib.sort_values(["rank", "cells"], ascending=[True, False])


In [ ]:
inner = (
    contrib.groupby("condition", observed=True)["cells"]
    .sum()
    .reindex([c for c in condition_order if c in set(contrib["condition"])])
)
inner_colors = [
    condition_palette.get(c, condition_palette.get(c.rstrip("+"), "lightgrey"))
    for c in inner.index
]
outer_colors = []


In [ ]:
for c, col in zip(inner.index, inner_colors):
    n = int((contrib["condition"] == c).sum())
    outer_colors += [to_rgba(col, alpha=a) for a in np.linspace(0.95, 0.35, n)]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))
ax.pie(inner.values,
    radius=0.72,
    colors=inner_colors,
    labels=inner.index,
    labeldistance=0.45,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=1),
    textprops=dict(fontsize=11, weight="bold", color="white"),
)
ax.pie(contrib["cells"].values,
    radius=1.0,
    colors=outer_colors,
    labels=contrib["sample_short"],
    labeldistance=1.03,
    rotatelabels=True,
    wedgeprops=dict(width=0.28, edgecolor="white", linewidth=0.6),
    textprops=dict(fontsize=6),
)
ax.set(aspect="equal")
ax.text(
    0,
    0,
    f"{epi.n_obs:,}\nepithelial cells\n{epi.obs['patient'].nunique()} patients",
    ha="center",
    va="center",
    fontsize=11,
)
save_figure(fig, fig_dir / "epithelial_contribution_by_sample", close=False, overwrite=False)
plt.show()


In [ ]:
contrib.to_csv(df_dir / "epithelial_contribution_per_sample.csv", index=False)


In [ ]:
epi_mask = adata.obs["epithelial_primary"].astype(bool)
x = np.asarray(adata[:, "EPCAM"].layers["log1p"].todense()).ravel() > 0


In [ ]:
print(f"EPCAM+ inside  epithelial : {100 * x[epi_mask].mean():.1f}%")
print(f"EPCAM+ outside epithelial : {100 * x[~epi_mask].mean():.1f}%")


## Integration diagnostics within cell types

In [ ]:
def broad_lineages(labels):
    """Avoid assessing mixing separately within arbitrary epithelial subtypes."""
    labels = labels.astype(str).copy()
    labels.loc[labels.str.startswith("Epithelial")] = "Epithelial"
    labels.loc[labels.isin(["T Cells", "NK Cells", "T/NK Cells"])] = "T/NK compartment"
    return labels


In [ ]:
def conditional_neighbour_mixing(
    obs, graph, annotation_key="cl_annotation", block_size=4096, collapse_lineages=True
):
    """Compare sample/donor neighbours within the same lineage AND condition.

    Expected same-unit probability excludes the focal cell. The expectation is
    a composition reference, not a calibrated null model or significance test.
    """
    required = [annotation_key, "condition", "sample", "patient"]
    if obs[required].isna().any().any():
        raise ValueError(
            "Complete annotation, condition, sample and patient metadata required"
        )
    if not obs.index.is_unique or graph.shape != (len(obs), len(obs)):
        raise ValueError("Graph must match unique metadata rows")
    if (
        (obs.groupby("sample", observed=True)[["patient", "condition"]].nunique() > 1)
        .any()
        .any()
    ):
        raise ValueError("A sample maps to multiple patients or conditions")
    frame = obs[required].copy()
    frame["lineage"] = (
        broad_lineages(frame[annotation_key])
        if collapse_lineages
        else frame[annotation_key].astype(str)
    )
    strata = ["lineage", "condition"]
    stratum_codes = pd.factorize(pd.MultiIndex.from_frame(frame[strata]))[0]
    group_n = (
        frame.groupby(strata, observed=True)["sample"].transform("size").to_numpy()
    )
    frame["n_cells_in_stratum"] = group_n
    sample_codes = pd.factorize(frame["sample"])[0]
    patient_codes = pd.factorize(frame["patient"])[0]
    g = graph.tocsr(copy=True)
    g.setdiag(0)
    g.eliminate_zeros()
    degree = np.zeros(len(frame), dtype=np.int64)
    same = {"sample": np.zeros(len(frame)), "patient": np.zeros(len(frame))}
    for start in range(0, len(frame), block_size):
        stop = min(start + block_size, len(frame))
        lo, hi = (g.indptr[start], g.indptr[stop])
        targets = g.indices[lo:hi]
        rows = np.repeat(np.arange(start, stop), np.diff(g.indptr[start : stop + 1]))
        valid = stratum_codes[rows] == stratum_codes[targets]
        rows, targets = (rows[valid], targets[valid])
        degree[start:stop] = np.bincount(rows - start, minlength=stop - start)
        for name, codes in [("sample", sample_codes), ("patient", patient_codes)]:
            same[name][start:stop] = np.bincount(
                rows - start,
                weights=codes[rows] == codes[targets],
                minlength=stop - start,
            )
    frame["n_eligible_neighbours"] = degree
    for name in ["sample", "patient"]:
        unit_n = (
            frame.groupby(strata + [name], observed=True)[name]
            .transform("size")
            .to_numpy()
        )
        n_units = (
            frame.groupby(strata, observed=True)[name].transform("nunique").to_numpy()
        )
        observed = np.divide(
            same[name], degree, out=np.full(len(frame), np.nan), where=degree > 0
        )
        expected = np.divide(
            unit_n - 1, group_n - 1, out=np.full(len(frame), np.nan), where=group_n > 1
        )
        frame[f"n_{name}s_in_stratum"] = n_units
        frame[f"observed_same_{name}"] = observed
        frame[f"expected_same_{name}"] = expected
        frame[f"excess_same_{name}"] = observed - expected
        frame[f"informative_{name}"] = (n_units > 1) & (degree >= 5)
    return frame


In [ ]:
if "X_scVI" not in adata.obsm or "connectivities" not in adata.obsp:
    raise ValueError("Run scVI and neighbours before these diagnostics")


In [ ]:
integration_out = Path(df_dir / "integration_diagnostics")
integration_out.mkdir(parents=True, exist_ok=True)


In [ ]:
integration_obs = adata.obs.copy()
if "cl_annotation" not in integration_obs:
    raise ValueError("Review cluster annotation before within-lineage mixing checks")
integration_obs["lineage"] = broad_lineages(integration_obs["cl_annotation"])


In [ ]:
integration_units = integration_obs[
    ["sample", "patient", "condition"]
].drop_duplicates()


In [ ]:
integration_units.to_csv(integration_out / "analysis_units.csv", index=False)


In [ ]:
for integration_key in ["sample", "patient", "condition"]:
    integration_counts = pd.crosstab(
        integration_obs["lineage"], integration_obs[integration_key]
    )
    integration_counts.to_csv(
        integration_out / f"lineage_by_{integration_key}_counts.csv"
    )
    integration_counts.div(integration_counts.sum(axis=1), axis=0).to_csv(
        integration_out / f"lineage_by_{integration_key}_fractions.csv"
    )


In [ ]:
integration_mixing = conditional_neighbour_mixing(
    integration_obs, adata.obsp["connectivities"], "cl_annotation"
)


In [ ]:
for integration_unit in ["sample", "patient"]:
    for integration_metric in ["observed", "excess"]:
        integration_column = f"{integration_metric}_same_{integration_unit}"
        integration_mixing[integration_column] = integration_mixing[
            integration_column
        ].where(integration_mixing[f"informative_{integration_unit}"])


In [ ]:
integration_per_library = integration_mixing.groupby(
    ["lineage", "condition", "sample", "patient"], observed=True
).agg(
    n_cells=("sample", "size"),
    median_eligible_neighbours=("n_eligible_neighbours", "median"),
    n_samples_in_stratum=("n_samples_in_stratum", "first"),
    n_patients_in_stratum=("n_patients_in_stratum", "first"),
    median_same_sample=("observed_same_sample", "median"),
    expected_same_sample=("expected_same_sample", "first"),
    median_excess_same_sample=("excess_same_sample", "median"),
    median_same_patient=("observed_same_patient", "median"),
    expected_same_patient=("expected_same_patient", "first"),
    median_excess_same_patient=("excess_same_patient", "median"),
    fraction_informative_sample=("informative_sample", "mean"),
)


In [ ]:
integration_per_library.to_csv(integration_out / "conditional_mixing_by_library.csv")


In [ ]:
integration_balanced = (
    integration_per_library.reset_index()
    .groupby(["lineage", "condition"], observed=True)
    .agg(
        n_libraries=("sample", "nunique"),
        n_patients=("patient", "nunique"),
        median_library_excess_same_sample=("median_excess_same_sample", "median"),
        median_library_excess_same_patient=("median_excess_same_patient", "median"),
        median_eligible_neighbours=("median_eligible_neighbours", "median"),
    )
)


In [ ]:
integration_balanced.to_csv(integration_out / "conditional_mixing_summary.csv")


In [ ]:
integration_subtype = conditional_neighbour_mixing(
    integration_obs,
    adata.obsp["connectivities"],
    "cl_annotation",
    collapse_lineages=False,
)


In [ ]:
for integration_unit in ["sample", "patient"]:
    integration_subtype[f"excess_same_{integration_unit}"] = integration_subtype[
        f"excess_same_{integration_unit}"
    ].where(integration_subtype[f"informative_{integration_unit}"])


In [ ]:
integration_subtype_library = integration_subtype.groupby(
    ["lineage", "condition", "sample", "patient"], observed=True
).agg(
    n_cells=("sample", "size"),
    median_excess_same_sample=("excess_same_sample", "median"),
    median_excess_same_patient=("excess_same_patient", "median"),
    fraction_informative_sample=("informative_sample", "mean"),
)


In [ ]:
integration_subtype_library.to_csv(
    integration_out / "annotated_subtype_mixing_by_library.csv"
)


In [ ]:
integration_subtype_summary = (
    integration_subtype_library.reset_index()
    .groupby(["lineage", "condition"], observed=True)
    .agg(
        n_libraries=("sample", "nunique"),
        n_patients=("patient", "nunique"),
        median_library_excess_same_sample=("median_excess_same_sample", "median"),
        median_library_excess_same_patient=("median_excess_same_patient", "median"),
    )
)


In [ ]:
integration_subtype_summary.to_csv(
    integration_out / "annotated_subtype_mixing_summary.csv"
)


In [ ]:
integration_latent = np.asarray(adata.obsm["X_scVI"])
integration_latent_columns = [
    f"latent_{integration_j + 1}"
    for integration_j in range(integration_latent.shape[1])
]


In [ ]:
integration_qc_columns = [
    integration_c
    for integration_c in [
        "total_counts",
        "n_genes_by_counts",
        "pct_counts_mt",
        "pct_counts_ribo",
        "doublet_score",
        "S_score",
        "G2M_score",
    ]
    if integration_c in integration_obs
]


In [ ]:
integration_groups = [("All cells", "All", np.arange(len(integration_obs)))]
integration_groups += [
    (str(integration_key[0]), str(integration_key[1]), integration_idx)
    for integration_key, integration_idx in integration_obs.groupby(
        ["lineage", "condition"], observed=True
    ).indices.items()
    if len(integration_idx) >= 200
]


In [ ]:
integration_rng = np.random.default_rng(0)


In [ ]:
integration_correlations = []


In [ ]:
for (
    integration_lineage,
    integration_condition,
    integration_indices,
) in integration_groups:
    integration_indices = np.asarray(integration_indices)
    integration_selected = (
        np.sort(integration_rng.choice(integration_indices, size=5000, replace=False))
        if len(integration_indices) > 5000
        else integration_indices
    )
    integration_data = pd.DataFrame(
        integration_latent[integration_selected], columns=integration_latent_columns
    )
    for integration_column in integration_qc_columns:
        integration_data[integration_column] = integration_obs.iloc[
            integration_selected
        ][integration_column].to_numpy()
    integration_corr = integration_data.corr(method="spearman").loc[
        integration_latent_columns, integration_qc_columns
    ]
    for integration_dim in integration_latent_columns:
        for integration_column in integration_qc_columns:
            integration_correlations.append(
                {
                    "lineage": integration_lineage,
                    "condition": integration_condition,
                    "latent_dimension": integration_dim,
                    "covariate": integration_column,
                    "spearman_r": integration_corr.loc[
                        integration_dim, integration_column
                    ],
                    "n_cells_total": len(integration_indices),
                    "n_cells_evaluated": len(integration_selected),
                }
            )


In [ ]:
integration_correlations = pd.DataFrame(integration_correlations)


In [ ]:
integration_correlations.to_csv(
    integration_out / "latent_qc_correlations.csv", index=False
)


In [ ]:
integration_cell_hash = hashlib.sha256(
    pd.util.hash_pandas_object(
        integration_obs[["cl_annotation", "sample", "patient", "condition"]], index=True
    )
    .to_numpy()
    .tobytes()
).hexdigest()


In [ ]:
(integration_out / "provenance.json").write_text(
    json.dumps(
        {
            "seed": 0,
            "annotation_key": "cl_annotation",
            "cell_metadata_sha256": integration_cell_hash,
            "n_cells": len(integration_obs),
            "n_latent": integration_latent.shape[1],
            "scope": "Descriptive post-training diagnostics; no model or annotation changes",
        },
        indent=2,
    )
)


In [ ]:
integration_review = {
    "mixing_by_library": integration_per_library,
    "mixing_summary": integration_balanced,
    "annotated_subtype_summary": integration_subtype_summary,
    "latent_qc_correlations": integration_correlations,
}
display(integration_review["annotated_subtype_summary"].round(3))


In [ ]:
session_info.show()
